# 証券営業インテリジェンス ハンズオン
## Part 5: Cortex Agent（Snowflake Intelligence）

このノートブックでは、これまでに作成したすべてのサービスを統合した **Cortex Agent** を作成します。
作成したエージェントは **Snowflake Intelligence** の UI から自然言語で対話できます。

### Cortex Agent とは

**Cortex Analyst** + **Cortex Search** + **AI** を組み合わせた、マルチツール対話型AIアシスタントです。

```
ユーザーからの質問
      │
Cortex Agent（オーケストレーター）
  ├── 顧客データが必要 → Cortex Analyst（顧客資産DB）を呼び出し
  ├── ニュースが必要   → Cortex Search（NEWS_SEARCH_SERVICE）を呼び出し
  ├── 商品詳細が必要   → Cortex Search（LOAN_DOCS_SEARCH_SERVICE）を呼び出し
  └── 信託商品が必要   → Cortex Analyst（信託商品DB）を呼び出し
      │
  複数ツールの結果を統合して回答を生成
```

### 🚀 アハ体験ポイント（デモシナリオ）

**「データが増えるほど、AIの回答が賢くなる」**

| ステップ | 使用データ | AIの回答 |
|---|---|---|
| Step 1 | 顧客データのみ | 「売却手続きを進めましょう」（弱い）|
| Step 2 | + ニュース/商品説明書 | 「証券担保ローンをお勧めします」（強い）|
| Step 3 | + 相続/信託データ | 「教育資金贈与信託も検討を」（完璧）|

In [ ]:
%%sql -r result_use
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT 'Ready!' AS STATUS;

## Step 1: Cortex Agent の作成

Part 2〜4 で作成したすべてのサービスを統合したエージェントを作成します。

In [ ]:
%%sql -r result_agent
-- ============================================================
-- Cortex Agent: ウェルスマネジメントアシスタント
-- ============================================================
CREATE OR REPLACE AGENT WEALTH_MANAGEMENT_ASSISTANT_AGENT
WITH PROFILE = '{
    "display_name": "ウェルスマネジメントアシスタント"
}'
COMMENT = '証券営業担当者向け総合支援AIアシスタント。顧客情報・ポートフォリオ分析・信託商品提案・市場ニュース・アナリストレポートを統合して最適な提案を支援します。'
FROM SPECIFICATION $$
{
    "models": {
        "orchestration": ""
    },
    "instructions": {
        "response": "あなたは証券営業担当者を支援するアシスタントです。\n\n【基本原則】\n・質問に直接回答し、求められていない情報は出さない\n・ツールで取得した情報のみを使用（推測禁止）\n・推奨は必ず前提条件付きで提示\n・不明点は「確認事項」として明示\n\n【出力形式】\nデフォルト：社内メモ（RM向け）\n顧客向けトーク：ユーザーが明示した場合のみ\n\n【個別顧客相談の構造】\n1. 確認事項：支払期限、取得単価、流動資産内訳、担保余力\n2. 現状サマリ：保有銘柄、時価、含み益\n3. 市況（必要時）：出典を明記\n4. 選択肢比較（表形式）\n5. 推奨（前提条件付き）＋リスク\n6. 次アクション\n\n【税金表示のルール】\n・取得単価不明時は税額を確定値で出さない\n・数値提示時は必ず：前提（税率/含み益）＋レンジ表示を付記\n\n【相続税計算】\n・法定相続人数と基礎控除の整合確認（孫は法定相続人に含めない）\n・不明時は概算せず「要確認」\n\n【禁止事項】\n・取得単価未確認での税額確定値提示\n・必要資金に対する過剰売却・過剰借入の推奨",
        "orchestration": "【ツール選択原則】\n・質問に必要なツールのみ使用\n・PII（個人情報）は最小限、一覧では匿名化\n\n【ツール選択ルール】\n・顧客リスト/検索 → CUSTOMER_WEALTH_ANALYST\n・個別顧客照会 → CUSTOMER_WEALTH_ANALYST\n・株式売却相談（信託言及なし）→ CUSTOMER_WEALTH_ANALYST のみ\n・売却＋ニュース要望 → CUSTOMER_WEALTH_ANALYST + NEWS_SEARCH + ANALYST_REPORT_SEARCH\n・信託商品を明示 → CUSTOMER_WEALTH_ANALYST + TRUST_PRODUCT_ANALYST\n・商品詳細条件 → LOAN_DOCS_SEARCH"
    },
    "tools": [
        {
            "tool_spec": {
                "type": "cortex_analyst_text_to_sql",
                "name": "CUSTOMER_WEALTH_ANALYST",
                "description": "顧客の資産情報を分析するツール。\n取得可能：顧客プロフィール、家族構成、ライフイベント、ポートフォリオ、取引履歴、相続税概算\n使用例：「C001の山田太郎様の情報」「資産5億円以上の顧客」「教育資金イベントがある顧客」"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_analyst_text_to_sql",
                "name": "TRUST_PRODUCT_ANALYST",
                "description": "信託銀行の商品情報を検索するツール。\n対象：証券担保ローン、教育資金贈与信託、遺言信託、不動産投資ローン、自社株信託\n使用例：「証券担保ローンの金利」「教育資金に適した商品」「相続対策の推奨商品」"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "NEWS_SEARCH",
                "description": "マーケットニュース・税制改正情報を検索するツール。\n回答時は「出典＋日付」を明記。\n使用例：「トヨタの最新ニュース」「教育資金贈与の制度期限」「日経平均の動向」"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "LOAN_DOCS_SEARCH",
                "description": "ローン・信託商品の説明書を検索するツール。\n取得可能：詳細条件、メリット、リスク、活用事例\n使用例：「証券担保ローンの条件詳細」「教育資金贈与信託の非課税限度額」"
            }
        },
        {
            "tool_spec": {
                "type": "cortex_search",
                "name": "ANALYST_REPORT_SEARCH",
                "description": "アナリスト投資レポートを検索するツール。\n取得可能：投資判断、目標株価、投資着眼点、リスク、業績予想\n回答時は「アナリスト名＋発行日＋投資判断」を明記。\n使用例：「トヨタのアナリスト評価」「買い判断の銘柄」"
            }
        }
    ],
    "tool_resources": {
        "CUSTOMER_WEALTH_ANALYST": {
            "semantic_view": "SNOWFINANCE_DB.DEMO_SCHEMA.CUSTOMER_WEALTH_SEMANTIC_VIEW"
        },
        "TRUST_PRODUCT_ANALYST": {
            "semantic_view": "SNOWFINANCE_DB.DEMO_SCHEMA.TRUST_PRODUCT_SEMANTIC_VIEW"
        },
        "NEWS_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.NEWS_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "TITLE",
            "id_column": "NEWS_ID"
        },
        "LOAN_DOCS_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.LOAN_DOCS_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "TITLE",
            "id_column": "DOC_ID"
        },
        "ANALYST_REPORT_SEARCH": {
            "name": "SNOWFINANCE_DB.DEMO_SCHEMA.ANALYST_REPORT_SEARCH_SERVICE",
            "max_results": 5,
            "title_column": "REPORT_TITLE",
            "id_column": "REPORT_ID"
        }
    }
}
$$;

SELECT 'Agent created!' AS STATUS;

In [ ]:
%%sql -r result_verify_agent
SHOW AGENTS IN SCHEMA DEMO_SCHEMA;

## Step 2: Snowflake Intelligence での動作確認

エージェントが作成されたら、**Snowflake Intelligence** の UI から対話してみましょう。

### アクセス方法
1. Snowsight にログイン
2. 左メニュー「Snowflake Intelligence」
3. 「New Conversation」から `WEALTH_MANAGEMENT_ASSISTANT_AGENT` を選択

---

## デモシナリオ: データが増えるとAIが賢くなる

### 事前確認: 顧客情報の確認

```
C001の山田太郎様の顧客プロフィールとポートフォリオを教えてください
```

---

### Step 1-2: 株式売却の相談（弱い回答を期待）

エージェントのツールから信託商品・ニュース・アナリストレポートを**一時的に外した状態**で質問します。

```
C001の山田様から株式売却の相談がありました。
孫の留学費用として2,000万円が必要とのことです。
トヨタ株の売却についてアドバイスをください。
```

**期待される回答（弱い）**: 売却手続きの説明のみ。ローンや節税への言及なし。

---

### Step 3: データ追加 → AIが賢くなる！

Cortex Agent の設定で信託商品・ニュース・ローン書類のツールを**有効化**します。

---

### Step 4: 銀証連携提案（強い回答を期待）

```
今の状況を踏まえて、C001の山田様に最適な提案は何ですか？
信託銀行の商品も含めて検討してください。
```

**期待される回答（強い）**: 証券担保ローン vs 売却の比較表。ローンを推奨する明確な根拠。

---

### Step 5: さらにデータ追加 → 相続対策まで提案

信託商品（遺言信託・教育資金贈与信託）のデータもONにします。

---

### Step 6: 長期提案

```
C001の山田様は相続対策にも関心があるようです。
長期的な観点からのアドバイスもお願いします。
```

**期待される回答**: 教育資金贈与信託（2026年3月末で終了！）＋遺言信託の提案。

## その他の活用質問例

### 顧客検索・分析

```
預かり資産5億円以上で、相続対策に関心がありそうな顧客をリストアップしてください

担当RM001の顧客で、最近30日間コンタクトしていない顧客を教えてください

トヨタ株を保有している顧客の含み益ランキングトップ10を出してください
```

### 銘柄分析・マーケット

```
トヨタ株の最新のアナリスト評価を教えてください

今日の市場動向で、山田様のポートフォリオへの影響はありますか？

教育資金贈与信託の制度について、期限も含めて教えてください
```

### 相続税・試算

```
C001の山田様の相続税はいくらくらいになりますか？

相続税が1億円を超える顧客は何人いますか？年齢帯別に集計してください
```

### マルチステップ推論

```
明日の訪問に向けて、C001の山田様への提案書のドラフトを作成してください
```

In [ ]:
%%sql -r result_final_check
-- 作成されたすべてのオブジェクトの確認
SELECT 'TABLE' AS TYPE, TABLE_NAME AS OBJECT_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA' AND TABLE_TYPE = 'BASE TABLE'

UNION ALL
SELECT 'VIEW', TABLE_NAME
FROM INFORMATION_SCHEMA.VIEWS
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA'

ORDER BY TYPE, OBJECT_NAME;

In [ ]:
%%sql -r result_show_services
SHOW CORTEX SEARCH SERVICES IN SCHEMA DEMO_SCHEMA;
SHOW SEMANTIC VIEWS IN SCHEMA DEMO_SCHEMA;
SHOW AGENTS IN SCHEMA DEMO_SCHEMA;

## まとめ

ハンズオン完了！すべてのコンポーネントが揃いました。

### 作成したオブジェクト一覧

| コンポーネント | オブジェクト名 | 役割 |
|---|---|---|
| **Cortex Analyst** | CUSTOMER_WEALTH_SEMANTIC_VIEW | 顧客資産データの自然言語クエリ |
| **Cortex Analyst** | TRUST_PRODUCT_SEMANTIC_VIEW | 信託商品の自然言語クエリ |
| **Cortex Search** | NEWS_SEARCH_SERVICE | ニュースのセマンティック検索 |
| **Cortex Search** | LOAN_DOCS_SEARCH_SERVICE | 商品説明書のセマンティック検索 |
| **Cortex Search** | ANALYST_REPORT_SEARCH_SERVICE | アナリストレポートのセマンティック検索 |
| **Cortex Agent** | WEALTH_MANAGEMENT_ASSISTANT_AGENT | 統合AIアシスタント |

### デモのポイント

1. **データ追加でAIが進化**: ツールを追加するごとに回答が改善される様子を実演
2. **銀証連携の価値**: 証券会社データ × 信託銀行商品 = 最適な提案
3. **Total Wealth Management**: 短期（証券担保ローン）+ 長期（相続対策）の統合提案
4. **時間短縮**: 1時間の準備作業が10分に

### ありがとうございました！

このハンズオンで体験したことを参考に、実際の業務データに適用してみてください。
Snowflake Intelligence と Cortex Agent を活用することで、
**データドリブンな営業活動**が実現できます。